# Ordered Logistic Regression Results for Adoption Predictors
This notebook provides a step-by-step example of loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.y7m0-f273) dataset using the [`mlcroissant`](https://mlcroissant.org/) library.

### Dataset Source
The dataset source is accessible via the Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install the mlcroissant library if not already present
!pip install mlcroissant

## 1. Data Loading
Load metadata and discover available record sets from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access and show dataset metadata
md = dataset.metadata
print(f"{md.name}: {md.description}")

## 2. Data Overview
List all available record sets, their `@id` identifiers, and the fields within each record set.

> **Note:** All references use `@id` strings (as specified by schema) to ensure robust, standards-compliant code.

In [ ]:
# List all record sets with their @id and fields
record_sets = {rs['@id']: rs for rs in md.to_json().get('recordSet', [])}

if not record_sets:
    print('No record sets defined in the dataset. Attempting to infer from resources...')
    # Try to infer record sets from DataDownload objects (distribution)
    for dist in md.to_json().get('distribution', []):
        print(f"- Data file @id: {dist['@id']}")
else:
    for rs_id, rs in record_sets.items():
        fields = rs.get('field', [])
        if isinstance(fields, dict):
            fields = [fields]
        field_ids = [f['@id'] for f in fields]
        print(f"Record set @id: {rs_id}\n  Fields: {', '.join(field_ids)}\n")

## 3. Data Extraction
Load data records from each record set (or data file) into pandas DataFrames.

> Here, datasets may use DataDownload/distribution if explicit record sets are not present. We iterate by all resources.

In [ ]:
# Attempt to extract data from all available record sets
dataframes = {}
# Use recordSets if present, otherwise try the distribution list as fallbacks (for typical Croissant datasets)
to_extract = list(record_sets) if record_sets else [dist['@id'] for dist in md.to_json().get('distribution', [])]

for record_set_id in to_extract:
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded DataFrame for record set @id: {record_set_id} with shape {df.shape}")
        else:
            print(f"No records found for {record_set_id}")
    except Exception as e:
        print(f"Could not load records for {record_set_id}: {e}")

# Print columns of first loaded DataFrame
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"\nColumns in DataFrame for {first_rs}:\n{dataframes[first_rs].columns.tolist()}")
    display(dataframes[first_rs].head())
else:
    print('No DataFrames loaded.')

## 4. Exploratory Data Analysis (EDA)
Perform basic processing: Filter records based on a numeric field, normalize data, and group by a categorical attribute if present.

**All fields and column references use their `@id` as per Croissant schema best practice.**

In [ ]:
# Pick the first available DataFrame and attempt to select useful fields
if dataframes:
    # Use first DataFrame
    rs_id = list(dataframes.keys())[0]
    df = dataframes[rs_id]
    print(f"\nWorking with record set: {rs_id}")
    numeric_field_id = None
    group_field_id = None

    # Try to select a numeric field (heuristically: any column with integer or float type)
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    # Try to select a group field (heuristically: object/String column with few unique values)
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and 2 < df[col].nunique() < max(10, 0.1 * len(df)):
            group_field_id = col
            break
    if numeric_field_id:
        threshold = df[numeric_field_id].median() if not pd.isnull(df[numeric_field_id].median()) else 0
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where `{numeric_field_id}` > {threshold}:")
        print(filtered_df.head())

        # Normalize the numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"\nNormalized `{numeric_field_id}` for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by 'group_field_id' if possible
        if group_field_id:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nMean `{numeric_field_id}` grouped by `{group_field_id}`:")
            print(grouped_df.head())
    else:
        print('No numeric field found for EDA.')
else:
    print('No data available for EDA.')

## 5. Visualization
Visualize the distribution of a numeric field and the mean per group if possible.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and numeric_field_id:
    fig, ax = plt.subplots(1, 2 if group_field_id else 1, figsize=(12, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True, ax=ax[0] if group_field_id else ax, color='skyblue')
    (ax[0] if group_field_id else ax).set_title(f'Distribution of {numeric_field_id}')
    if group_field_id:
        if 'grouped_df' in locals():
            sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, ax=ax[1], palette='dark')
            ax[1].set_title(f'Mean {numeric_field_id} by {group_field_id}')
            ax[1].tick_params(axis='x', rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('Insufficient data to plot.')

## 6. Conclusion
This notebook demonstrated how to load, inspect, and perform basic analysis on the FAIR^2 dataset using its Croissant schema definition and the `mlcroissant` Python library.

- The dataset metadata describes ordered logistic regression results for household knowledge adoption regarding rangeland management interventions in Northern Kenya.
- Record sets (or data files) were programmatically discovered and loaded for analysis.
- Numeric fields were explored, filtered, and visualized to identify distributional patterns and groupwise effects (where grouping fields existed).

Further analysis can explore more dataset relationships, variable documentation (see Croissant schema), or custom analytic questions relevant to climate adaptation and gender inclusion in rangeland management.